# NutriMatch — Feature Engineering: Kategori, Bahan Dasar & Meal Time
**Tujuan notebook ini:**
Menambahkan 3 fitur baru ke dalam `food_master_clean.csv`.

**Input:** `food_master_clean.csv`  
**Output:** `food_master_terbari.csv`

---
**Pipeline:**
1. Import & Load
2. Definisi Keyword Rules
3. Labeling `cooking_category`
4. Labeling `main_ingredient`
5. Labeling `meal_time` (multilabel)
6. Audit Coverage
7. Export

## 1. Import Library

In [17]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


## 2. Load Data

In [18]:
df = pd.read_csv('food_master_clean.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

Shape: (1332, 11)
Columns: ['food_id', 'food_name', 'calories_100g', 'protein_100g', 'fat_100g', 'carbohydrate_100g', 'source', 'image_url', 'zero_calorie_flag', 'calorie_inconsistent_flag', 'calorie_macro_diff']


,food_id,food_name,calories_100g,protein_100g,fat_100g,carbohydrate_100g,source,image_url,zero_calorie_flag,calorie_inconsistent_flag,calorie_macro_diff
0,FD0001,abon,280.0,9.2,28.4,0.0,indonesian_food_nutrition_dataset,https://img-cdn.medkomtek.com/PbrY9X3ignQ8sVuj...,0,0,-12.4
1,FD0002,abon haruwan,513.0,23.7,37.0,21.3,indonesian_food_nutrition_dataset,https://img-global.cpcdn.com/recipes/cbf330fbd...,0,0,0.0
2,FD0003,agar-agar,0.0,0.0,0.2,0.0,indonesian_food_nutrition_dataset,https://res.cloudinary.com/dk0z4ums3/image/upl...,1,0,-1.8
3,FD0004,akar tonjong segar,45.0,1.1,0.4,10.8,indonesian_food_nutrition_dataset,https://images.tokopedia.net/img/cache/200-squ...,0,0,-6.2
4,FD0005,aletoge segar,37.0,4.4,0.5,3.8,indonesian_food_nutrition_dataset,https://nilaigizi.com/assets/images/produk/pro...,0,0,-0.3


## 3. Definisi Keyword Rules

Strategi: priority-based matching — urutan dict menentukan prioritas.  
Satu makanan bakal dapat **satu label** untuk `cooking_category` dan `main_ingredient`

Untuk `meal_time`: multilabel, satu makanan bisa match lebih dari 1 kategori.

In [19]:
# Cooking Category Rules
COOKING_RULES = {
    'gorengan'  : ['goreng'],
    'bakar'     : ['bakar', 'panggang'],
    'berkuah'   : ['soto', 'sop', 'sup', 'rawon', 'bakso', 'kuah',
                   'gulai', 'opor', 'lodeh', 'kari', 'garang asem'],
    'tumis'     : ['tumis', 'oseng', 'cah'],
    'rebus_kukus': ['rebus', 'kukus', 'tim', 'pepes', 'pindang'],
    'sambal'    : ['sambal'],
    'olahan'    : ['abon', 'nugget', 'sosis', 'kornet', 'dendeng',
                   'kerupuk', 'keripik', 'chips', 'snack', 'masakan',
                   'burger', 'pizza', 'sandwich', 'pasta'],
    'mentah_segar': ['segar', 'mentah'],
    'other'     : []  # fallback
}

# Main Ingredient Rules
INGREDIENT_RULES = {
    'ayam'    : ['ayam'],
    'sapi'    : ['sapi', 'daging sapi'],
    'babi'    : ['babi'],
    'kambing' : ['kambing'],
    'ikan'    : ['ikan', 'salmon', 'tuna', 'lele', 'nila', 'bandeng',
                 'cakalang', 'tongkol', 'gabus', 'gurame', 'kakap'],
    'seafood' : ['udang', 'cumi', 'kerang', 'lobster', 'kepiting', 'sotong'],
    'telur'   : ['telur', 'telor'],
    'kedelai' : ['tahu', 'tempe', 'kedelai', 'oncom'],
    'sayuran' : ['bayam', 'kangkung', 'wortel', 'buncis', 'kacang panjang',
                 'kubis', 'brokoli', 'labu', 'terong', 'sawi', 'selada',
                 'timun', 'paprika', 'jagung', 'pete', 'jengkol',
                 'nangka', 'rebung', 'jamur'],
    'buah'    : ['apel', 'pisang', 'mangga', 'jeruk', 'melon', 'semangka',
                 'anggur', 'nanas', 'pepaya', 'alpukat', 'durian', 'salak',
                 'leci', 'manggis', 'rambutan', 'markisa', 'sirsak',
                 'belimbing', 'jambu', 'kedondong', 'bengkuang'],
    'beras'   : ['nasi', 'beras', 'lontong', 'ketupat', 'bubur nasi',
                 'liwet', 'ketan'],
    'terigu'  : ['roti', 'mie', 'pasta', 'biscuit', 'kue', 'cake',
                 'martabak', 'risol', 'lumpia', 'bakpao', 'donat'],
    'singkong': ['singkong', 'tape', 'getuk', 'ubi kayu'],
    'susu'    : ['susu', 'keju', 'yogurt', 'krim', 'mentega'],
    'kacang'  : ['kacang tanah', 'kacang hijau', 'kacang merah',
                 'kacang mede', 'almond', 'kenari'],
    'other'   : []  # fallback
}

# Meal Time Rules (MULTILABEL)
# Satu makanan bisa match lebih dari satu waktu makan (kaya nasi bisa buat lunch sama dinner)
MEALTIME_RULES = {
    'breakfast': ['bubur', 'roti', 'susu', 'jus', 'teh', 'kopi',
                  'sereal', 'oatmeal', 'pancake', 'toast', 'sandwich',
                  'cornflakes', 'yogurt', 'granola'],
    'lunch'    : ['nasi', 'lontong', 'ketupat', 'rawon', 'soto', 'sop',
                  'gulai', 'rendang', 'semur', 'gado', 'pecel', 'rujak',
                  'lotek', 'ketoprak', 'mie', 'bakso', 'sate'],
    'dinner'   : ['nasi', 'rawon', 'rendang', 'sop', 'gulai', 'semur',
                  'sup', 'soto', 'ikan bakar', 'ayam bakar', 'grilled',
                  'steak', 'mie goreng']
}

print('Rules defined:')
print(f'  cooking_category : {len(COOKING_RULES)} kategori')
print(f'  main_ingredient  : {len(INGREDIENT_RULES)} kategori')
print(f'  meal_time        : {len(MEALTIME_RULES)} kategori (multilabel)')

Rules defined:
  cooking_category : 9 kategori
  main_ingredient  : 16 kategori
  meal_time        : 3 kategori (multilabel)


## 4. Labeling `cooking_category`

In [20]:
def label_cooking(food_name: str) -> str:
    for category, keywords in COOKING_RULES.items():
        if category == 'other':
            continue
        if any(kw in food_name for kw in keywords):
            return category
    return 'other'

df['cooking_category'] = df['food_name'].apply(label_cooking)

print('Distribusi cooking_category:')
print(df['cooking_category'].value_counts().to_string())
print(f'\nTotal : {len(df):,}')
print(f'Labeled (bukan other): {(df["cooking_category"] != "other").sum()} ({(df["cooking_category"] != "other").mean()*100:.1f}%)')
print(f'Other  : {(df["cooking_category"] == "other").sum()} ({(df["cooking_category"] == "other").mean()*100:.1f}%)')

Distribusi cooking_category:
cooking_category
other           718
mentah_segar    322
olahan           88
gorengan         76
rebus_kukus      73
berkuah          38
tumis             9
bakar             8

Total : 1,332
Labeled (bukan other): 614 (46.1%)
Other  : 718 (53.9%)


## 5. Labeling `main_ingredient`

In [21]:
def label_ingredient(food_name: str) -> str:
    for ingredient, keywords in INGREDIENT_RULES.items():
        if ingredient == 'other':
            continue
        if any(kw in food_name for kw in keywords):
            return ingredient
    return 'other'

df['main_ingredient'] = df['food_name'].apply(label_ingredient)

print('Distribusi main_ingredient:')
print(df['main_ingredient'].value_counts().to_string())
print(f'\nTotal : {len(df):,}')
print(f'Labeled (bukan other): {(df["main_ingredient"] != "other").sum()} ({(df["main_ingredient"] != "other").mean()*100:.1f}%)')

Distribusi main_ingredient:
main_ingredient
other       661
ikan        138
sayuran      94
buah         88
kedelai      58
beras        52
terigu       41
ayam         37
sapi         35
kacang       30
singkong     26
susu         21
seafood      19
telur        14
babi         13
kambing       5

Total : 1,332
Labeled (bukan other): 671 (50.4%)


## 6. Labeling `meal_time` (Multilabel)

Makanan di Indonesia banyak yang bisa dimakan kapan saja. Jadinya ngegunakan multilabel: satu makanan bisa `breakfast`, `lunch`, `dinner`, atau kombinasi.

Format output: string dipisah koma, contoh `lunch,dinner`  
Default fallback: `lunch,dinner` (paling umum buat makanan Indonesia)

In [22]:
def label_meal_time(food_name: str) -> str:
    matched = []
    for meal, keywords in MEALTIME_RULES.items():
        if any(kw in food_name for kw in keywords):
            if meal not in matched:
                matched.append(meal)

    # Deduplicate (nasi match di lunch dan dinner jadinya ['lunch','dinner'])
    if not matched:
        return 'lunch,dinner'
    return ','.join(sorted(set(matched)))

df['meal_time'] = df['food_name'].apply(label_meal_time)

print('Distribusi meal_time:')
print(df['meal_time'].value_counts().head(15).to_string())
print()
print('Breakdown per slot:')
for slot in ['breakfast','lunch','dinner']:
    count = df['meal_time'].str.contains(slot).sum()
    print(f'  {slot:12s}: {count:,} makanan')

Distribusi meal_time:
meal_time
lunch,dinner        1232
dinner,lunch          49
breakfast             29
lunch                 21
breakfast,dinner       1

Breakdown per slot:
  breakfast   : 30 makanan
  lunch       : 1,302 makanan
  dinner      : 1,282 makanan


## 7. Audit Coverage

In [23]:
print('COVERAGE AUDIT — Feature Engineering')
print('-' * 50)

total = len(df)

# cooking_category
cc_labeled = (df['cooking_category'] != 'other').sum()
print(f'\ncooking_category')
print(f'  Labeled  : {cc_labeled:,} ({cc_labeled/total*100:.1f}%)')
print(f'  Other    : {total-cc_labeled:,} ({(total-cc_labeled)/total*100:.1f}%)')

# main_ingredient
mi_labeled = (df['main_ingredient'] != 'other').sum()
print(f'\nmain_ingredient')
print(f'  Labeled  : {mi_labeled:,} ({mi_labeled/total*100:.1f}%)')
print(f'  Other    : {total-mi_labeled:,} ({(total-mi_labeled)/total*100:.1f}%)')

# meal_time
mt_explicit = (df['meal_time'] != 'lunch,dinner').sum()
print(f'\nmeal_time')
print(f'  Explicit label   : {mt_explicit:,} ({mt_explicit/total*100:.1f}%)')
print(f'  Default fallback : {total-mt_explicit:,} ({(total-mt_explicit)/total*100:.1f}%)')

print()
print('Sample hasil labeling')
df[['food_name','cooking_category','main_ingredient','meal_time']].sample(15, random_state=42)

COVERAGE AUDIT — Feature Engineering
--------------------------------------------------

cooking_category
  Labeled  : 614 (46.1%)
  Other    : 718 (53.9%)

main_ingredient
  Labeled  : 671 (50.4%)
  Other    : 661 (49.6%)

meal_time
  Explicit label   : 100 (7.5%)
  Default fallback : 1,232 (92.5%)

Sample hasil labeling


,food_name,cooking_category,main_ingredient,meal_time
1280,tiwul,other,other,"lunch,dinner"
1061,santan murni,other,other,"lunch,dinner"
1250,tepung sente,other,other,"lunch,dinner"
298,daun singkong kopang segar,mentah_segar,singkong,"lunch,dinner"
237,daun gandaria,other,other,"lunch,dinner"
588,kacang belimbing kecipir kering,other,buah,"lunch,dinner"
240,daun gunda bali segar,mentah_segar,other,"lunch,dinner"
277,daun oyong,other,other,"lunch,dinner"
887,mie ayam,other,ayam,lunch
704,kembang tahu,other,kedelai,"lunch,dinner"


## 8. Export `food_master_enriched.csv`

Output ini menambahkan 3 kolom baru ke dalan`food_master_clean.csv`

In [24]:
# food_master_clean terbaru
OUTPUT_COLS = df.columns.tolist()

food_master_enriched = df.copy()

print('FOOD MASTER ENRICHED — SUMMARY')
print('-' * 50)
print(f'Total rows    : {len(food_master_enriched):,}')
print(f'Total columns : {len(food_master_enriched.columns)}')
print(f'New columns   : cooking_category, main_ingredient, meal_time')
print()
print('Kolom lengkap:')
print(food_master_enriched.columns.tolist())

food_master_enriched.to_csv('food_master_terbaru.csv', index=False)
print('\nDisimpan: food_master_terbaru.csv')

FOOD MASTER ENRICHED — SUMMARY
--------------------------------------------------
Total rows    : 1,332
Total columns : 14
New columns   : cooking_category, main_ingredient, meal_time

Kolom lengkap:
['food_id', 'food_name', 'calories_100g', 'protein_100g', 'fat_100g', 'carbohydrate_100g', 'source', 'image_url', 'zero_calorie_flag', 'calorie_inconsistent_flag', 'calorie_macro_diff', 'cooking_category', 'main_ingredient', 'meal_time']

Disimpan: food_master_terbaru.csv
